In [ ]:
import pandas as pd
import numpy as np
import re 

MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]
MONTH_NUM = {m: i + 1 for i, m in enumerate(MONTHS)}

def parse_saws(file_path):
    saws_raw = pd.read_excel(file_path, sheet_name = "DETAIL001", header= None)
    records = []
    current_measure = None
    current_station = {}
    current_obs_time = None 
    
    for i, row in saws_raw.iterrows():
        vals = [str(v) if pd.notna(v) else "" for v in row.values]
        first = vals[0].strip()
        
        if re.match(r"^(Maximum|Minimum|Total cloud)", first, re.IGNORECASE):
            if "Maximum" in first:
                current_measure = "max_temp_c"
            elif "Minimum" in first:
                current_measure = "min_temp_c"
            elif "cloud" in first:
                current_measure = "cloud_octas"
            continue 
    
        if "Data for station" in first:
            if "Maximum" in first:
                current_measure = "max_temp_c"
            elif "Minimum" in first:
                current_measure = "min_temp_c"
            elif "Cloud" in first:
                current_measure = "cloud_octas"
                
            m = re.search(
                r"\[(.+?)\]\s*-\s*(.+?)\s+([-\d.]+)\s+([-\d.]+)\s+([\d.]+)\s*m\s+(\d{4})"
                r".*?(\d{2}:\d{2})",
                first
            )
            
            if m:
                sid, name, lat, lon, elev, year, obs_time = m.groups()
            else:
                m = re.search(
                    r"\[(.+?)\]\s*-\s*(.+?)\s+([-\d.]+)\s+([-\d.]+)\s+([\d.]+)\s*m\s+(\d{4})",
                    first 
                )
                if m: 
                    sid, name, lat, lon, elev, year = m.groups()
                    obs_time = "unkown"
                    
            if m:
                current_station = {
                    "station_id": sid.strip(),
                    "station_name": name.strip().title(),
                    "lat": float(lat),
                    "lon": float(lon),
                    "elevation_m": float(elev),
                    "year": int(year),
                }
                current_obs_time = obs_time 
            continue 
        
        if first.startswith("---") or first.startswith("Avg") or first == "Day":
            continue
        
        try:
            day = int(float(first))
        except (ValueError, TypeError):
            continue 
        
        if not (1 <= day <= 31) or not current_station or not current_measure:
            continue 
        
        for col_index, month_name in enumerate(MONTHS):
            raw = vals[col_index + 1].strip() if col_index + 1 < len(vals) else ""
            
            try:
                value = float(raw)
            except ValueError:
                value = np.nan 
                
            try:
                date = pd.Timestamp(
                    year = current_station["year"],
                    month = MONTH_NUM[month_name],
                    day = day
                )
            except ValueError:
                continue 
            
            records.append({
                "station_id": current_station["station_id"],
                "station_name": current_station["station_name"],
                "lat": current_station["lat"],
                "lon": current_station["lon"],
                "elevation_m": current_station["elevation_m"],
                "year": current_station["year"],
                "date": date,
                "obs_time": current_obs_time,
                "measure": current_measure,
                "value": value,
            })
        
    return pd.DataFrame(records)

In [ ]:
saws1 = parse_saws("Country stations 2021-26 data1.xlsx")
print(saws1.shape)
print(saws1["measure"].unique())
print(saws1["station_name"].unique()) 

In [ ]:
saws2 = parse_saws("Country stations 2021-26 data2.xlsx")
print(saws2.shape)
print(saws2["measure"].unique())
print(saws2["obs_time"].unique())

In [ ]:
saws2_daily = (
   saws2.groupby(["station_id", "station_name", "lat", "lon", "elevation_m", "year", "date","measure", ])["value"].mean().reset_index()
)
saws2_daily["obs_time"] = "daily_avg"

In [ ]:
saws = pd.concat([saws1, saws2_daily], ignore_index = True)
saws.to_csv("saws_2021_2026.csv", index = False)
print(saws.head())